# Instruction embeddings, English (CheckThat! 2026 Task 2)
### Qwen3-Embedding-8B · claim + evidence (3 instructions) · claim + all reasoning traces (1 instruction) · resumable · Kaggle 2×T4

The existing English vectors were made **without** an instruction (`CLAIM_INSTRUCTION = None`). Qwen3-Embedding is
instruction-aware: a text is embedded as `Instruct: <task>\nQuery:<text>`. This notebook re-embeds the same rows
with task instructions, so that "instruction vs no instruction" is the only difference.

| Stage | Texts | Weights | Output file (one per split) | Format |
|:--|:--|:--|:--|:--|
| `ce_instrA/B/C` | claim + **all** evidence, one text per row | 4-bit NF4, one copy per GPU | `{split}_claimevid_qwen3emb8b4bit_en_instrA.npz` (B, C) | same as `..._en_v1.npz`: `emb (N,4096)`, `__ids__`, `ntok`, `truncated` |
| `traces_instr` | claim (row 0) + every non-empty trace | fp16 split over both GPUs | `{split}_qwen3emb8b_en_instr_v1.npz` | same as `qwen3emb8b_en_v1`: `id_<id>` → `(1+n_traces, 4096)` |

The weights of each stage match the existing no-instruction file it is compared with (claim + evidence was 4-bit,
traces were fp16). Evidence is kept in full (`MAX_SEQ_LENGTH = 8192`; the longest test row is about 5.5k tokens).

**Resume.** Work is saved in shards of 64 rows. On start, every merged file or shard with the same settings found in
`OUTPUT_DIR` or under `RESUME_SEARCH_ROOT` is checked and reused. The run stops cleanly after
`EMBED_TIME_BUDGET_HOURS`. To continue: save the version, add its output as an input, run again.
The claim + evidence stages run first (short), the trace stage last (long: about 210k texts).

**Run on Kaggle:** accelerator **GPU T4 ×2**, **internet on**, *Save & Run All*.

## 1. Configuration

In [1]:
from collections import OrderedDict

# ─────────────────────────── PATHS ───────────────────────────
DATA_DIR           = "/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/English"
OUTPUT_DIR         = "/kaggle/working"
RESUME_SEARCH_ROOT = "/kaggle/input"       # searched recursively for earlier shards / merged files

# ─────────────────────────── WHAT TO RUN ───────────────────────────
RUN_CLAIM_EVIDENCE = True
RUN_TRACES         = True

# ─────────────────────────── MODEL ───────────────────────────
MODEL_ID       = "Qwen/Qwen3-Embedding-8B"
MAX_SEQ_LENGTH = 8192
CE_QUANT       = "4bit"      # as the existing claimevid_qwen3emb8b4bit_en_v1 vectors
TRACE_QUANT    = "fp16"      # as the existing qwen3emb8b_en_v1 claim / trace vectors
GPU_MAX_MEMORY = {0: "12GiB", 1: "12GiB"}    # fp16 only: weights split over the two T4s

# ─────────────────────────── INSTRUCTIONS ───────────────────────────
# claim + evidence: one output file per key. Edit / add / remove freely; a changed text needs a new key.
CE_INSTRUCTIONS = OrderedDict([
    ("instrA", "Given a numerical claim and retrieved evidence, classify whether the evidence shows the claim "
               "to be True, False or Conflicting"),
    ("instrB", "Verify the numbers, quantities and dates stated in the claim against the evidence and decide "
               "whether the claim is supported, refuted or only partly supported"),
    ("instrC", "Given a claim and evidence, determine whether the evidence is sufficient and consistent, or "
               "mixed and conflicting, for verifying the claim"),
])
# claim + traces: row 0 (claim) and rows 1.. (traces) each get their own instruction
TRACE_TAG               = "qwen3emb8b_en_instr_v1"
TRACE_CLAIM_INSTRUCTION = "Given a numerical claim, represent it for classifying it as True, False or Conflicting"
TRACE_INSTRUCTION       = ("Given a reasoning trace that fact-checks a numerical claim, identify the verdict it "
                           "argues for and how well its reasoning supports that verdict")

# ─────────────────────────── TEXT TEMPLATE (same as the existing claim + evidence vectors) ───────────────────────────
CLAIM_PREFIX    = "Claim: "
EVIDENCE_PREFIX = "Evidence {i}: "
SEPARATOR       = "\n\n"

# ─────────────────────────── THROUGHPUT / RESUME ───────────────────────────
TOKEN_BUDGET            = 16384     # max padded tokens (batch × longest) per forward pass
MAX_TEXTS_PER_BATCH     = 64
SHARD_SIZE              = 64        # rows (claims) per resumable shard
EMBED_TIME_BUDGET_HOURS = 11.0      # no new shard is started after this (Kaggle limit is 12 h)
DELETE_SHARDS_AFTER_MERGE = True
STORE_DTYPE             = "float32"
LIMIT_ROWS              = None      # e.g. 200 for a quick trial run (writes files tagged '_trial'); None = everything

# ─────────────────────────── DATA FIELDS ───────────────────────────
CLAIM_FIELD, EVIDENCE_FIELD, TRACE_FIELD, TEST_ID_FIELD = "claim", "evidences", "Reasoning_traces", "query_id"

## 2. Setup

In [2]:
import os, re, gc, sys, json, glob, math, time, queue, shutil, threading, warnings, subprocess, importlib.util
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
T_START = time.time()
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"PyTorch : {torch.__version__} | CUDA {torch.cuda.is_available()}")
for _i in range(torch.cuda.device_count()):
    _p = torch.cuda.get_device_properties(_i)
    print(f"  GPU {_i}: {_p.name}  {_p.total_memory / 1e9:.1f} GB")

PyTorch : 2.11.0+cu130 | CUDA True
  GPU 0: Tesla T4  15.6 GB
  GPU 1: Tesla T4  15.6 GB


## 3. Data and texts
IDs are the same as in the existing embedding files: position in the file for train / validation, `query_id` for
test. All rows are kept (no deduplication).

In [3]:
SPLIT_FILES = OrderedDict(train="train.json", val="validation.json", test="test.json")

def find_one(root, rel):
    if not root or not os.path.isdir(root):
        return None
    hits = [h for h in glob.glob(os.path.join(root, "**", rel), recursive=True)
            if "arab" not in h.lower() and "span" not in h.lower()]
    return min(hits, key=len) if hits else None

def load_json(path):
    with open(path, encoding="utf-8") as f:
        s = f.read()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        print(f"  {os.path.basename(path)}: invalid JSON, repairing the missing 'Reasoning_traces' key")
        return json.loads(re.sub(r'\{\s*\nx?(\s+)"(?![A-Za-z_]+"\s*:)', r'{\n        "Reasoning_traces": [\n\1"', s))

def with_instruction(text, instruction):
    return text if instruction is None else f"Instruct: {instruction}\nQuery:{text}"

def evidence_list(item):
    ev = item.get(EVIDENCE_FIELD) or []
    if isinstance(ev, str):
        ev = [ev]
    return [str(e).strip() for e in ev if str(e).strip()]

def claim_evidence_text(item):
    parts = [CLAIM_PREFIX + item[CLAIM_FIELD].strip()]
    parts += [EVIDENCE_PREFIX.format(i=i) + e for i, e in enumerate(evidence_list(item), 1)]
    return SEPARATOR.join(parts)

def trace_texts(item):
    return [t for t in item[TRACE_FIELD] if isinstance(t, str) and t.strip()]

DATA = OrderedDict()
for split, fname in SPLIT_FILES.items():
    path = find_one(DATA_DIR, fname)
    if path is None:
        raise FileNotFoundError(f"{fname} not found under {DATA_DIR}: attach the data and fix DATA_DIR in §1")
    items = load_json(path)
    ids = [it[TEST_ID_FIELD] for it in items] if split == "test" else list(range(len(items)))
    if len(set(ids)) != len(ids):
        raise ValueError(f"{split}: duplicate ids")
    by_id = dict(zip(ids, items))
    order = sorted(ids)
    if LIMIT_ROWS:
        order = order[:LIMIT_ROWS]
    DATA[split] = {"path": path, "order": order, "by_id": by_id}
    n_tr = [len(trace_texts(by_id[i])) for i in order]
    print(f"{split:5s}: {len(order):5d} rows | traces/row mean {np.mean(n_tr):.2f} | "
          f"rows without evidence {sum(len(evidence_list(by_id[i])) == 0 for i in order)} | {path}")

_sfx = "_trial" if LIMIT_ROWS else ""
STAGES = []
if RUN_CLAIM_EVIDENCE:
    for key, instr in CE_INSTRUCTIONS.items():
        tag = f"claimevid_qwen3emb8b{'4bit' if CE_QUANT == '4bit' else ''}_en_{key}{_sfx}"
        STAGES.append({
            "name": f"ce_{key}", "kind": "ce", "tag": tag, "quant": CE_QUANT,
            "texts": (lambda it, instr=instr: [with_instruction(claim_evidence_text(it), instr)]),
            "meta": {"tag": tag, "model": MODEL_ID, "quantization": CE_QUANT, "max_seq_length": MAX_SEQ_LENGTH,
                     "instruction": instr, "format": "Instruct: {instruction}\\nQuery:{text}",
                     "template": {"claim": CLAIM_PREFIX, "evidence": EVIDENCE_PREFIX, "separator": SEPARATOR},
                     "pooling": "last_token", "l2_normalized": True,
                     "truncation": "right (end of the evidence), end token kept", "store_dtype": STORE_DTYPE}})
if RUN_TRACES:
    tag = TRACE_TAG + _sfx
    STAGES.append({
        "name": "traces_instr", "kind": "traces", "tag": tag, "quant": TRACE_QUANT,
        "texts": (lambda it: [with_instruction(it[CLAIM_FIELD].strip(), TRACE_CLAIM_INSTRUCTION)] +
                             [with_instruction(t, TRACE_INSTRUCTION) for t in trace_texts(it)]),
        "meta": {"tag": tag, "model": MODEL_ID, "quantization": TRACE_QUANT, "max_seq_length": MAX_SEQ_LENGTH,
                 "claim_instruction": TRACE_CLAIM_INSTRUCTION, "trace_instruction": TRACE_INSTRUCTION,
                 "format": "Instruct: {instruction}\\nQuery:{text}", "pooling": "last_token", "l2_normalized": True,
                 "store_dtype": STORE_DTYPE, "trace_field": TRACE_FIELD,
                 "row_layout": "row0=claim, rows1..n=non-empty reasoning traces in original order"}})
for st in STAGES:
    print(f"stage {st['name']:13s} → {{split}}_{st['tag']}.npz  [{st['quant']}]")
_it = DATA["train"]["by_id"][DATA["train"]["order"][0]]
for st in STAGES[:1] + STAGES[-1:]:
    _t = st["texts"](_it)
    print(f"\n[{st['name']}] first train row, {len(_t)} text(s); last text starts:\n{_t[-1][:420]}...")

train:  6400 rows | traces/row mean 19.85 | rows without evidence 166 | /kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/English/train.json
val  :  1600 rows | traces/row mean 15.00 | rows without evidence 42 | /kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/English/validation.json
test :  2558 rows | traces/row mean 20.00 | rows without evidence 0 | /kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/English/test.json
stage ce_instrA     → {split}_claimevid_qwen3emb8b4bit_en_instrA.npz  [4bit]
stage ce_instrB     → {split}_claimevid_qwen3emb8b4bit_en_instrB.npz  [4bit]
stage ce_instrC     → {split}_claimevid_qwen3emb8b4bit_en_instrC.npz  [4bit]
stage traces_instr  → {split}_qwen3emb8b_en_instr_v1.npz  [fp16]

[ce_instrA] first train row, 1 text(s); last text starts:
Instruct: Given a numerical claim and retrieved evidence, classify whether the evidence shows the claim to be True, False or Conflicting
Query:Claim: “The first randomized c

## 4. Shards, merged files and resume
A shard holds the vectors of `SHARD_SIZE` rows (`emb`, with `counts` = vectors per row). It is reused only if its
metadata equals the stage's settings, its IDs and counts are the expected ones, and all vectors are finite unit
vectors.

In [4]:
def shard_plan(split):
    o = DATA[split]["order"]
    return [(k, o[k * SHARD_SIZE:(k + 1) * SHARD_SIZE]) for k in range(math.ceil(len(o) / SHARD_SIZE))]

def shard_path(st, split, k): return f"{OUTPUT_DIR}/emb_shards/{st['tag']}/{split}/shard_{k:05d}.npz"
def final_name(st, split):    return f"{split}_{st['tag']}.npz"
def final_path(st, split):    return f"{OUTPUT_DIR}/{final_name(st, split)}"

def expected_counts(st, split, ids):
    by_id = DATA[split]["by_id"]
    return [1 if st["kind"] == "ce" else 1 + len(trace_texts(by_id[i])) for i in ids]

def _meta_ok(z, st):
    return "__meta__" in z.files and json.loads(str(z["__meta__"])) == st["meta"]

def read_shard(st, split, k, ids):
    path = shard_path(st, split, k)
    if not os.path.exists(path):
        return None
    try:
        with np.load(path, allow_pickle=False) as z:
            if not _meta_ok(z, st) or list(z["__ids__"]) != [str(i) for i in ids]:
                return None
            emb, counts = z["emb"], z["counts"]
            if list(counts) != expected_counts(st, split, ids) or emb.shape[0] != counts.sum():
                return None
            if not np.isfinite(emb).all() or np.abs(np.linalg.norm(emb.astype(np.float32), axis=1) - 1).max() > 1e-3:
                return None
            return {"emb": emb, "counts": counts, "ntok": z["ntok"], "truncated": z["truncated"]}
    except Exception as e:
        print(f"  ! unreadable {path}: {e}")
        return None

def save_shard(st, split, k, ids, emb, counts, ntok, trunc):
    path = shard_path(st, split, k)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path[:-4] + ".tmp.npz"
    np.savez(tmp, __meta__=np.array(json.dumps(st["meta"], ensure_ascii=False)),
             __ids__=np.array([str(i) for i in ids]), emb=emb.astype(STORE_DTYPE),
             counts=np.asarray(counts, np.int32), ntok=np.asarray(ntok, np.int32), truncated=np.asarray(trunc, bool))
    os.replace(tmp, path)

def check_final(st, split, path):
    """True iff a merged file has the stage's settings and exactly the expected rows."""
    order = DATA[split]["order"]
    try:
        with np.load(path, allow_pickle=False) as z:
            if not _meta_ok(z, st) or list(z["__ids__"]) != [str(i) for i in order]:
                return False
            if st["kind"] == "ce":
                return z["emb"].shape[0] == len(order)
            cnt = expected_counts(st, split, order)
            keys = set(z.files)
            if any(f"id_{i}" not in keys for i in order):
                return False
            probe = [0, len(order) // 2, len(order) - 1]          # full check is done once at merge time
            return all(z[f"id_{order[r]}"].shape[0] == cnt[r] for r in probe)
    except Exception as e:
        print(f"  ! unreadable {path}: {e}")
        return False

def merge_split(st, split):
    plan = shard_plan(split)
    parts = [read_shard(st, split, k, ids) for k, ids in plan]
    if any(p is None for p in parts):
        return False
    order = DATA[split]["order"]
    tmp = final_path(st, split)[:-4] + ".tmp.npz"
    common = dict(__meta__=np.array(json.dumps(st["meta"], ensure_ascii=False)),
                  __ids__=np.array([str(i) for i in order]))
    if st["kind"] == "ce":
        np.savez(tmp, emb=np.concatenate([p["emb"] for p in parts]).astype(STORE_DTYPE),
                 ntok=np.concatenate([p["ntok"] for p in parts]),
                 truncated=np.concatenate([p["truncated"] for p in parts]), **common)
    else:
        arrays = {}
        for (k, ids), p in zip(plan, parts):
            off = np.concatenate([[0], np.cumsum(p["counts"])])
            for j, i in enumerate(ids):
                arrays[f"id_{i}"] = p["emb"][off[j]:off[j + 1]].astype(STORE_DTYPE)
        np.savez(tmp, **arrays, **common)
        del arrays
    os.replace(tmp, final_path(st, split))
    if not check_final(st, split, final_path(st, split)):
        raise RuntimeError(f"[{st['name']} / {split}] merged file failed verification")
    if DELETE_SHARDS_AFTER_MERGE:
        shutil.rmtree(os.path.dirname(shard_path(st, split, 0)), ignore_errors=True)
    print(f"[{st['name']} / {split}] merged → {final_path(st, split)} "
          f"({os.path.getsize(final_path(st, split)) / 1e6:.0f} MB)", flush=True)
    return True

def import_previous(st):
    """Reuses merged files / shards of this stage from OUTPUT_DIR or RESUME_SEARCH_ROOT."""
    done, copied = set(), 0
    search = bool(RESUME_SEARCH_ROOT) and os.path.isdir(RESUME_SEARCH_ROOT)
    for split in DATA:
        cands = [final_path(st, split)]
        if search:
            cands += sorted(glob.glob(f"{RESUME_SEARCH_ROOT}/**/{final_name(st, split)}", recursive=True), key=len)
        for p in cands:
            if os.path.exists(p) and check_final(st, split, p):
                if os.path.abspath(p) != os.path.abspath(final_path(st, split)):
                    shutil.copy(p, final_path(st, split))
                    print(f"[{st['name']} / {split}] merged file reused ← {p}")
                done.add(split)
                break
        if split in done or not search:
            continue
        plan = dict(shard_plan(split))
        for p in glob.glob(f"{RESUME_SEARCH_ROOT}/**/emb_shards/{st['tag']}/{split}/shard_*.npz", recursive=True):
            m = re.search(r"shard_(\d+)\.npz$", p)
            k = int(m.group(1)) if m else -1
            dst = shard_path(st, split, k)
            if k in plan and not os.path.exists(dst):
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copy(p, dst)
                if read_shard(st, split, k, plan[k]) is None:
                    os.remove(dst)
                else:
                    copied += 1
    return done, copied

for st in STAGES:
    st["done"], n_copied = import_previous(st)
    st["pending"] = []
    status = {}
    for split in DATA:
        if split in st["done"]:
            status[split] = "merged"
            continue
        plan = shard_plan(split)
        todo = [(split, k, ids) for k, ids in plan if read_shard(st, split, k, ids) is None]
        st["pending"] += todo
        status[split] = f"{len(plan) - len(todo)}/{len(plan)} shards"
    print(f"{st['name']:13s}: {status} | shards imported {n_copied} | to embed {len(st['pending'])}")

ce_instrA    : {'train': '0/100 shards', 'val': '0/25 shards', 'test': '0/40 shards'} | shards imported 0 | to embed 165
ce_instrB    : {'train': '0/100 shards', 'val': '0/25 shards', 'test': '0/40 shards'} | shards imported 0 | to embed 165
ce_instrC    : {'train': '0/100 shards', 'val': '0/25 shards', 'test': '0/40 shards'} | shards imported 0 | to embed 165
traces_instr : {'train': '0/100 shards', 'val': '0/25 shards', 'test': '0/40 shards'} | shards imported 0 | to embed 165


## 5. Model
`4bit`: one NF4 copy per GPU, the two GPUs work in parallel. `fp16`: one copy split over both GPUs.
The tokenizer's end token is kept after truncation and is the pooled (last) position; padding is on the left.

In [5]:
ENCODERS, LOADED_QUANT, EOS_SUFFIX, PAD_ID, EMB_DIM, DTYPE_KW = [], None, None, None, None, "torch_dtype"

def detect_suffix(tok):
    probe = "Claim: test 123"
    with_sp = tok(probe)["input_ids"]
    plain = tok(probe, add_special_tokens=False)["input_ids"]
    if len(with_sp) > len(plain) and with_sp[:len(plain)] == plain:
        return with_sp[len(plain):]
    end = tok.get_vocab().get("<|endoftext|>", tok.eos_token_id)
    if end is None:
        raise ValueError("the tokenizer appends no end token and has no <|endoftext|>/eos: cannot last-token pool")
    return [end]

def tokenize(tok, texts):
    keep = MAX_SEQ_LENGTH - len(EOS_SUFFIX)
    plain = tok(texts, add_special_tokens=False)["input_ids"]
    return [p[:keep] + EOS_SUFFIX for p in plain], [len(p) > keep for p in plain]

def make_batches(lengths):
    order = sorted(range(len(lengths)), key=lambda i: -lengths[i])   # longest first: an OOM shows up at once
    batches, cur, cur_max = [], [], 0
    for i in order:
        new_max = max(cur_max, lengths[i])
        if cur and (new_max * (len(cur) + 1) > TOKEN_BUDGET or len(cur) >= MAX_TEXTS_PER_BATCH):
            batches.append(cur)
            cur, new_max = [], lengths[i]
        cur.append(i)
        cur_max = new_max
    if cur:
        batches.append(cur)
    return batches

class Encoder:
    def __init__(self, quant, gpu):
        from transformers import AutoModel, AutoTokenizer
        self.gpu = gpu
        self.tok = AutoTokenizer.from_pretrained(MODEL_ID)
        kw = dict(attn_implementation="sdpa", low_cpu_mem_usage=True, **{DTYPE_KW: torch.float16})
        if quant == "4bit":
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_use_double_quant=True,
                                                           bnb_4bit_compute_dtype=torch.float16)
            kw["device_map"] = {"": gpu}
        elif torch.cuda.device_count() > 1:
            kw["device_map"] = "auto"
            kw["max_memory"] = {k: v for k, v in GPU_MAX_MEMORY.items() if k < torch.cuda.device_count()}
        else:
            kw["device_map"] = {"": 0}
        self.model = AutoModel.from_pretrained(MODEL_ID, **kw).eval()
        self.in_dev = self.model.get_input_embeddings().weight.device

    @torch.inference_mode()
    def _forward(self, seqs):
        L = max(len(s) for s in seqs)
        ids = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
        mask = torch.zeros((len(seqs), L), dtype=torch.long)
        for r, s in enumerate(seqs):                     # left padding: last position = end token
            ids[r, L - len(s):] = torch.tensor(s)
            mask[r, L - len(s):] = 1
        h = self.model(input_ids=ids.to(self.in_dev), attention_mask=mask.to(self.in_dev),
                       use_cache=False).last_hidden_state[:, -1].float()
        finite = torch.isfinite(h).all(dim=1)
        return F.normalize(h, p=2, dim=-1).cpu().numpy(), finite.cpu().numpy()

    def encode(self, seqs):
        """OOM → split the batch in two; a non-finite row is re-encoded alone, and the run stops if it stays non-finite."""
        oom = False
        try:
            emb, finite = self._forward(seqs)
        except torch.cuda.OutOfMemoryError:
            oom = True
        if oom:
            torch.cuda.empty_cache()
            if len(seqs) == 1:
                raise RuntimeError(f"OOM on one {len(seqs[0])}-token text: lower MAX_SEQ_LENGTH (and change the tags)")
            h = len(seqs) // 2
            return np.concatenate([self.encode(seqs[:h]), self.encode(seqs[h:])])
        for j in np.where(~finite)[0]:
            e1, f1 = self._forward([seqs[j]])
            if not f1.all():
                raise RuntimeError(f"non-finite embedding for a {len(seqs[j])}-token text even when encoded alone")
            emb[j] = e1[0]
        return emb

    def embed_texts(self, texts):
        seqs, trunc = tokenize(self.tok, texts)
        out = np.empty((len(seqs), EMB_DIM), np.float32)
        for b in make_batches([len(s) for s in seqs]):
            out[b] = self.encode([seqs[i] for i in b])
        return out, np.array([len(s) for s in seqs], np.int32), np.array(trunc, bool)

def free_encoders():
    global ENCODERS, LOADED_QUANT
    for e in ENCODERS:
        del e.model
    ENCODERS, LOADED_QUANT = [], None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def ensure_encoders(quant):
    global ENCODERS, LOADED_QUANT, EOS_SUFFIX, PAD_ID, EMB_DIM, DTYPE_KW
    if LOADED_QUANT == quant:
        return
    free_encoders()
    if not torch.cuda.is_available():
        raise RuntimeError("no GPU: switch the accelerator to GPU T4 x2")
    if quant == "4bit" and importlib.util.find_spec("bitsandbytes") is None:
        print("installing bitsandbytes ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"], check=True)
    import transformers
    transformers.logging.set_verbosity_error()
    _v = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
    DTYPE_KW = "dtype" if _v >= (4, 56) else "torch_dtype"
    n = torch.cuda.device_count() if quant == "4bit" else 1
    for g in range(n):
        t = time.time()
        ENCODERS.append(Encoder(quant, g))
        print(f"[{quant}] model copy {g} loaded in {time.time() - t:.0f}s | " +
              " | ".join(f"GPU {i}: {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB"
                         for i in range(torch.cuda.device_count())), flush=True)
    tok = ENCODERS[0].tok
    EOS_SUFFIX = detect_suffix(tok)
    PAD_ID = tok.pad_token_id if tok.pad_token_id is not None else EOS_SUFFIX[-1]
    EMB_DIM = ENCODERS[0].model.config.hidden_size
    LOADED_QUANT = quant
    print(f"end token(s) {tok.convert_ids_to_tokens(EOS_SUFFIX)} | pad id {PAD_ID} | hidden size {EMB_DIM}")

## 6. Embed
Stages run in order. Each shard is saved as soon as it is done, so an interrupted run loses at most the shard in
progress. No new shard starts after `EMBED_TIME_BUDGET_HOURS`.

In [6]:
DEADLINE = T_START + EMBED_TIME_BUDGET_HOURS * 3600

def run_stage(st):
    jobs_list = st["pending"]
    jobs = queue.Queue()
    for j in jobs_list:
        jobs.put(j)
    stop, lock = threading.Event(), threading.Lock()
    s = {"done": 0, "tok": 0, "rows": 0, "errors": [], "t0": time.time()}
    total_rows = sum(len(j[2]) for j in jobs_list)

    def worker(enc):
        while not stop.is_set() and time.time() < DEADLINE:
            try:
                split, k, ids = jobs.get_nowait()
            except queue.Empty:
                return
            t = time.time()
            try:
                per_item = [st["texts"](DATA[split]["by_id"][i]) for i in ids]
                emb, ntok, trunc = enc.embed_texts([x for p in per_item for x in p])
                save_shard(st, split, k, ids, emb, [len(p) for p in per_item], ntok, trunc)
            except Exception as e:
                with lock:
                    s["errors"].append(f"{st['name']} {split} shard {k}: {type(e).__name__}: {e}")
                stop.set()
                return
            with lock:
                s["done"] += 1; s["tok"] += int(ntok.sum()); s["rows"] += len(ids)
                el = time.time() - s["t0"]
                eta = (total_rows - s["rows"]) / max(s["rows"] / el, 1e-9) / 60
                if s["done"] % 5 == 1 or s["done"] == len(jobs_list):
                    print(f"[{st['name']}] {split:5s} shard {k:4d} | {len(ntok):5d} texts | {ntok.sum() / 1e3:7.1f}k tok | "
                          f"trunc {int(trunc.sum())} | {time.time() - t:6.1f}s | {s['done']}/{len(jobs_list)} | "
                          f"{s['tok'] / el:,.0f} tok/s | ETA {eta:.0f} min | "
                          f"session {(time.time() - T_START) / 3600:.2f} h", flush=True)

    threads = [threading.Thread(target=worker, args=(e,), daemon=True) for e in ENCODERS]
    for t in threads: t.start()
    for t in threads: t.join()
    if s["errors"]:
        raise RuntimeError("embedding stopped:\n  " + "\n  ".join(s["errors"]))
    return len(jobs_list) - s["done"]

for st in STAGES:
    if st["pending"] and time.time() < DEADLINE:
        print(f"\n===== {st['name']}: {len(st['pending'])} shard(s), {sum(len(j[2]) for j in st['pending'])} rows =====")
        ensure_encoders(st["quant"])
        left = run_stage(st)
        if left:
            print(f"[{st['name']}] time budget reached with {left} shard(s) left")
    for split in DATA:
        if split not in st["done"] and merge_split(st, split):
            st["done"].add(split)
free_encoders()


===== ce_instrA: 165 shard(s), 10558 rows =====
installing bitsandbytes ...


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (6 outputs left out)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[4bit] model copy 1 loaded in 15s | GPU 0: 4.5 GiB | GPU 1: 4.5 GiB
end token(s) ['<|endoftext|>'] | pad id 151643 | hidden size 4096
[ce_instrA] train shard    1 |    64 texts |    14.9k tok | trunc 0 |   20.9s | 1/165 | 710 tok/s | ETA 57 min | session 0.05 h
[ce_instrA] train shard    5 |    64 texts |    15.4k tok | trunc 0 |   38.3s | 6/165 | 1,091 tok/s | ETA 37 min | session 0.07 h
[ce_instrA] train shard    9 |    64 texts |    15.4k tok | trunc 0 |   30.6s | 11/165 | 1,142 tok/s | ETA 34 min | session 0.09 h
[ce_instrA] train shard   15 |    64 texts |    14.7k tok | trunc 0 |   24.1s | 16/165 | 1,093 tok/s | ETA 34 min | session 0.11 h
[ce_instrA] train shard   20 |    64 texts |    13.3k tok | trunc 0 |   26.1s | 21/165 | 1,117 tok/s | ETA 32 min | session 0.13 h
[ce_instrA] train shard   25 |    64 texts |    14.5k tok | trunc 0 |   24.5s | 26/165 | 1,086 tok/s | ETA 32 min | session 0.15 h
[ce_instrA] train shard   30 |    64 texts |    14.1k tok | trunc 0 |   25.8s | 31/1

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[fp16] model copy 0 loaded in 10s | GPU 0: 8.0 GiB | GPU 1: 6.1 GiB
end token(s) ['<|endoftext|>'] | pad id 151643 | hidden size 4096
[traces_instr] train shard    0 |  1342 texts |   415.3k tok | trunc 0 |  480.4s | 1/165 | 865 tok/s | ETA 1313 min | session 4.99 h
[traces_instr] train shard    5 |  1344 texts |   400.6k tok | trunc 0 |  464.6s | 6/165 | 862 tok/s | ETA 1211 min | session 5.62 h
[traces_instr] train shard   10 |  1344 texts |   367.8k tok | trunc 0 |  426.0s | 11/165 | 864 tok/s | ETA 1153 min | session 6.23 h
[traces_instr] train shard   15 |  1344 texts |   393.9k tok | trunc 0 |  461.7s | 16/165 | 861 tok/s | ETA 1125 min | session 6.87 h
[traces_instr] train shard   20 |  1344 texts |   381.5k tok | trunc 0 |  444.4s | 21/165 | 860 tok/s | ETA 1082 min | session 7.48 h
[traces_instr] train shard   25 |  1344 texts |   359.2k tok | trunc 0 |  420.1s | 26/165 | 860 tok/s | ETA 1042 min | session 8.11 h
[traces_instr] train shard   30 |  1344 texts |   382.9k tok | t

## 7. Summary

In [7]:
rows, all_done = [], True
for st in STAGES:
    for split in DATA:
        r = {"stage": st["name"], "split": split, "rows": len(DATA[split]["order"]),
             "status": "done" if split in st["done"] else "NOT FINISHED"}
        if split in st["done"]:
            r["file"] = final_name(st, split)
            with np.load(final_path(st, split), allow_pickle=False) as z:
                if st["kind"] == "ce":
                    r.update({"vectors": z["emb"].shape[0], "dim": z["emb"].shape[1],
                              "tokens p50": int(np.median(z["ntok"])), "tokens max": int(z["ntok"].max()),
                              "truncated": int(z["truncated"].sum())})
                else:
                    a = z[f"id_{DATA[split]['order'][0]}"]
                    r.update({"vectors": int(sum(expected_counts(st, split, DATA[split]["order"]))), "dim": a.shape[1]})
        else:
            all_done = False
        rows.append(r)
display(pd.DataFrame(rows).astype(object).fillna(""))
with open(f"{OUTPUT_DIR}/instruction_embeddings_meta.json", "w", encoding="utf-8") as f:
    json.dump({"stages": [{"name": st["name"], "meta": st["meta"], "done": sorted(st["done"])} for st in STAGES]},
              f, indent=2, ensure_ascii=False)
if all_done:
    print("✓ all stages finished. Add this notebook's output as an input of the fine-tuning + ablation notebook.")
else:
    print("✗ not finished. Save this version, add its output as an input of this notebook, and run again.")
print(f"total time {(time.time() - T_START) / 60:.1f} min")

,stage,split,rows,status,file,vectors,dim,tokens p50,tokens max,truncated
0,ce_instrA,train,6400,done,train_claimevid_qwen3emb8b4bit_en_instrA.npz,6400.0,4096.0,198.0,2324.0,0.0
1,ce_instrA,val,1600,done,val_claimevid_qwen3emb8b4bit_en_instrA.npz,1600.0,4096.0,231.0,4844.0,0.0
2,ce_instrA,test,2558,done,test_claimevid_qwen3emb8b4bit_en_instrA.npz,2558.0,4096.0,2244.0,5553.0,0.0
3,ce_instrB,train,6400,done,train_claimevid_qwen3emb8b4bit_en_instrB.npz,6400.0,4096.0,203.0,2329.0,0.0
4,ce_instrB,val,1600,done,val_claimevid_qwen3emb8b4bit_en_instrB.npz,1600.0,4096.0,236.0,4849.0,0.0
5,ce_instrB,test,2558,done,test_claimevid_qwen3emb8b4bit_en_instrB.npz,2558.0,4096.0,2249.0,5558.0,0.0
6,ce_instrC,train,6400,done,train_claimevid_qwen3emb8b4bit_en_instrC.npz,6400.0,4096.0,199.0,2325.0,0.0
7,ce_instrC,val,1600,done,val_claimevid_qwen3emb8b4bit_en_instrC.npz,1600.0,4096.0,232.0,4845.0,0.0
8,ce_instrC,test,2558,done,test_claimevid_qwen3emb8b4bit_en_instrC.npz,2558.0,4096.0,2245.0,5554.0,0.0
9,traces_instr,train,6400,NOT FINISHED,,,,,,


✗ not finished. Save this version, add its output as an input of this notebook, and run again.
total time 665.9 min
